<a href="https://colab.research.google.com/github/diliprc96/TrafficFlowOptimization/blob/dev/Traffic_Flow_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### `---------------Mandatory Information to fill------------`

### Group ID:
### Group Members Name with Student ID:
1. Student 1
2. Student 2
3. Student 3
4. Student 4


`-------------------Write your remarks (if any) that you want should get consider at the time of evaluation---------------`

Remarks: ##Add here

## Objective:
Develop and compare Reinforcement Learning agents (DQN and Actor-Critic) to optimize traffic flow and vehicle speed regulation. Implement RL-based strategies to enhance traffic efficiency, reduce congestion, and improve safety by adaptively controlling the speed and lane changes of a selected vehicle within the simulated environment.




## Dataset:

***Link for accessing dataset:*** https://drive.google.com/file/d/1pyExKzpKVRhFr2Ltfp6Ts8yF8OdWZNgH/view?usp=drive_link

**Time Step:** The dataset provides vehicle trajectory data at a frequency of 10 Hz, meaning each frame represents a 0.1-second interval.

## State Space :

The state represents the current traffic conditions and vehicle status:

1. Vehicle Speed (v_Vel) (m/s)

2.  Vehicle Acceleration (v_Acc) (m/s²)

3.  Lane Position (Lane_ID)

4.  Distance to Preceding Vehicle (Space_Headway) (m)

5.  Time Gap to Preceding Vehicle (Time_Headway) (s)

6.  Vehicle Class (v_Class)

7.  Global X (Global_X)

8.  Global Y (Global_Y)

Total State Vector Dimension: 8 features


## Action Space :

| Action | Description         | Conditions                                                | Change Applied  |
|--------|---------------------|----------------------------------------------------------|-----------------|
| 0      | Maintain current speed | No change required                                      | 0 m/s adjustment |
| 1      | Increase speed      | If Space_Headway ≥ 15m                                  | +2 m/s          |
| 2      | Decrease speed      | If Space_Headway < 10m                                  | −2 m/s          |
| 3      | Change to left lane | If left lane exists and is not occupied and Space_Headway ≥ 15m | Move left       |
| 4      | Change to right lane | If right lane exists and is not occupied and Space_Headway ≥ 15m | Move right      |


## Traffic Safety and Target Speed:

**Safe Following Distance:** At least 15 meters from the preceding vehicle *(Space_Headway≥15m)*.

**Collision Risk:** Less than 5 meters gap is unsafe *(Space_Headway<5m)*.

**Optimal Target Speed:** 27 m/s (approximately 60 mph, highway recommended speed).

## Reward Function:

\begin{equation}
R = (10 − |V_t - V_{\text{optimal}}| ) − P_{\text{collision}}
\end{equation}

Where:

- $ V_t $ = Current vehicle speed (m/s)  
- $ V_{\text{optimal}} $ = **27 m/s** (highway optimal speed)  
- $ P_{\text{collision}} $  =  
  \begin{cases}
  20, & \text{if SpaceHeadway} < 5m \text{ (high collision risk)} \\
  0, & \text{otherwise}
  \end{cases}




## Requirements and Deliverables:

Implement the Traffic Flow Optimization Problem for the given above scenario for all the below mentioned RL methods.

### Initialize constants

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [8]:
# Constants
State_representaton_features = ['v_Vel', 'v_Acc', 'Lane_ID', 'Space_Headway', 'Time_Headway', 'v_Class', 'Global_X', 'Global_Y']

### Load Dataset    (1 Mark)

In [9]:
# Code for Dataset loading and preprocessing
#-----write your code below this line---------

Raw_Traffic_data = pd.read_csv('/content/drive/MyDrive/Traffic_flow_optimization/DQN_DDQN_Actor_Critic_assignment.csv')
Raw_Traffic_data.info()

# Convert relevant columns (e.g., v_Vel, v_Acc, Space_Headway, Time_Headway, etc.) to numerical data types.

# Normalize or standardize the numerical data to improve training stability.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Vehicle_ID     20000 non-null  int64  
 1   Frame_ID       20000 non-null  int64  
 2   Total_Frames   20000 non-null  int64  
 3   Global_Time    20000 non-null  int64  
 4   Local_X        20000 non-null  float64
 5   Local_Y        20000 non-null  float64
 6   Global_X       20000 non-null  float64
 7   Global_Y       20000 non-null  float64
 8   v_length       20000 non-null  float64
 9   v_Width        20000 non-null  float64
 10  v_Class        20000 non-null  int64  
 11  v_Vel          20000 non-null  float64
 12  v_Acc          20000 non-null  float64
 13  Lane_ID        20000 non-null  int64  
 14  O_Zone         20000 non-null  float64
 15  D_Zone         20000 non-null  float64
 16  Int_ID         20000 non-null  float64
 17  Section_ID     20000 non-null  float64
 18  Direct

In [10]:
Traffic_data = Raw_Traffic_data[State_representaton_features]
Traffic_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   v_Vel          20000 non-null  float64
 1   v_Acc          20000 non-null  float64
 2   Lane_ID        20000 non-null  int64  
 3   Space_Headway  20000 non-null  float64
 4   Time_Headway   20000 non-null  float64
 5   v_Class        20000 non-null  int64  
 6   Global_X       20000 non-null  float64
 7   Global_Y       20000 non-null  float64
dtypes: float64(6), int64(2)
memory usage: 1.2 MB


In [25]:
Traffic_data.head()

,v_Vel,v_Acc,Lane_ID,Space_Headway,Time_Headway,v_Class,Global_X,Global_Y
0,18.52,-10.02,2,456.15,24.63,2,2230819.012,1377022.878
1,26.56,12.27,1,104.06,3.92,2,2230614.927,1376331.219
2,38.40,-1.26,1,203.40,5.30,2,2230824.438,1376973.785
3,35.91,0.51,11,0.00,0.00,2,6452175.016,1873370.389
4,0.00,0.00,1,0.00,0.00,2,6452418.326,1874274.428


### Design a Traffic Control Environment (0.5 Mark)

In [ ]:
# Code for environment creation
#-----write your code below this line---------

class TrafficEnvironment:

### Define Action Functions: (2.5 Mark)

1.  MaintainSpeed
2.  IncreaseSpeed
3.  DecreaseSpeed
4.  ChangeLaneLeft
5.  ChangeLaneRight


In [ ]:
# Code for action function
#-----write your code below this line-------

def MaintainSpeed():
  #-----
def IncreaseSpeed():
  #-----
def DecreaseSpeed():
  #-----
def ChangeLaneLeft():
  #-----
def ChangeLaneRight():
  #-----

### Implement the Reward Function  (1 Mark)


In [ ]:
# Code for reward function
#-----write your code below this line-------

def Reward():
  #-------

### Implement a Replay Buffer for experience storage in DQN   (1 Mark)

In [ ]:
# Code for replay buffer
#-----write your code below this line-------

### Design and Train DQN   (2.5 Mark)


In [ ]:
# Code for training
#-----write your code below this line------

### Design and train Actor-Critic Algorithm  (2.5 Mark)


In [ ]:
# Code for training
#-----write your code below this line------

### Plot the graph for Average Reward   (1 Mark)

In [ ]:
# Code for plotting the average reward
#-----write your code below this line------

### Compare and summarize Traffic Flow Outcomes for DQN  and Actor Critic  (1 Mark)
